# StainDetector Playground

This notebook provides examples for exploring data, visualizing patches, and testing model components.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision import transforms
import seaborn as sns

# Add the src directory to path
sys.path.append(os.path.join(".."))

# Import project modules
from src.dataset import StainDataset
from src.model import StainClassifier, get_model
from src.config import load_config
from src.utils import set_seed, extract_patches_from_wsi

# Set random seed for reproducibility
set_seed(42)

## 1. Load and Explore Configuration

In [ ]:
# Load configuration
config = load_config("../configs/train_config.yaml")

# Display configuration
print(config)

## 2. Explore Dataset

In [ ]:
# Create dataset (adjust paths if needed)
train_dataset = StainDataset(
    data_dir="../data/train",
    image_size=224,
    mode="train"
)

# Display dataset information
print(f"Number of samples: {len(train_dataset)}")
print(f"Classes: {train_dataset.classes}")

# Count samples per class
class_counts = {}
for label in train_dataset.labels:
    class_name = train_dataset.classes[label]
    class_counts[class_name] = class_counts.get(class_name, 0) + 1

# Plot class distribution
plt.figure(figsize=(10, 6))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()))
plt.title("Class Distribution in Training Data")
plt.xlabel("Stain Class")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Visualize Some Samples

In [ ]:
# Visualize a few samples
def show_samples(dataset, num_samples=5):
    plt.figure(figsize=(15, 3*num_samples))
    for i in range(num_samples):
        # Get a sample
        idx = np.random.randint(0, len(dataset))
        img, label = dataset[idx]
        
        # Convert to numpy and denormalize if needed
        if isinstance(img, torch.Tensor):
            img = img.numpy()
            img = np.transpose(img, (1, 2, 0))
            # Denormalize (assuming ImageNet normalization)
            img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img = np.clip(img, 0, 1)
        
        class_name = dataset.classes[label]
        plt.subplot(num_samples, 1, i+1)
        plt.imshow(img)
        plt.title(f"Class: {class_name}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Show some samples
show_samples(train_dataset, num_samples=5)

## 4. Test Model Creation

In [ ]:
# Create a model using the configuration
model = StainClassifier(config)

# Print model summary
print(model)

# Test forward pass
dummy_input = torch.randn(1, 3, config.data.image_size, config.data.image_size)
output = model(dummy_input)
print(f"Output shape: {output.shape}")

## 5. Explore WSI Processing (if available)

In [ ]:
# Path to a WSI file (update with actual path if available)
wsi_path = "../data/example_wsis/sample.svs"

# Extract patches if WSI file exists
if os.path.exists(wsi_path):
    patches = extract_patches_from_wsi(
        wsi_path=wsi_path,
        patch_size=config.inference_defaults.patch_size,
        num_patches=10,  # Small number for testing
        tissue_threshold=config.inference_defaults.tissue_threshold
    )
    
    # Display extracted patches
    plt.figure(figsize=(15, 10))
    for i, patch in enumerate(patches[:5]):
        plt.subplot(1, 5, i+1)
        plt.imshow(patch)
        plt.title(f"Patch {i+1}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"WSI file not found: {wsi_path}")